In [1]:
import pandas as pd
import numpy as np
import pickle
import os
import gc
from rank_bm25 import BM25Okapi

Khai báo đường dẫn local

In [2]:
data_path = "data/news_dataset_df_prep2.pkl"

save_dir = "indexes/bm25"
os.makedirs(save_dir, exist_ok=True)

print("Data path:", data_path)
print("Save dir:", save_dir)

Data path: data/news_dataset_df_prep2.pkl
Save dir: indexes/bm25


Load dữ liệu

In [3]:
try:
    df_load = pd.read_pickle(data_path)

    print("DataFrame loaded successfully.")
    print("Loaded DataFrame shape:", df_load.shape)
    print("Columns:")
    print(df_load.columns)

except FileNotFoundError:
    print(f"Error: File not found at {data_path}")

except Exception as e:
    print(f"Error loading DataFrame: {e}")

DataFrame loaded successfully.
Loaded DataFrame shape: (160254, 14)
Columns:
Index(['id', 'author', 'content', 'picture_count', 'processed', 'source',
       'title', 'topic', 'url', 'crawled_at', 'title_processed',
       'content_processed', 'combined_processed', 'combined_unaccented'],
      dtype='object')


Chọn cột văn bản cho BM25

In [4]:
text_column = "combined_processed"

documents = df_load[text_column].fillna("").astype(str).tolist()

print("Số lượng document:", len(documents))
print("Ví dụ document đầu tiên:")
print(documents[0][:500])

Số lượng document: 160254
Ví dụ document đầu tiên:
tên cướp tiệm vàng tại huế là đại_uý công_an công_tác tại trại_giam chiều date0731 công_an tỉnh thừa_thiên - huế đã có thông_tin ban_đầu về vụ nổ_súng cướp tiệm vàng tại chợ đông_ba nằm trên đường trần_hưng_đạo tp huế tỉnh thừa_thiên - huế_thông_sài_gòn giải_phóng khoảng 12h30 ngày date0731 một đối_tượng sử_dụng súng ak bất_ngờ xông vào tiệm vàng hoàng_đức và thái_lợi phía trước chợ đông_ba rồi nổ_súng chỉ_thiên liên_tiếp uy_hiếp chủ tiệm để cướp vàng sau đó đối_tượng mang số vàng vừa cướp được 


Tokenize corpus

In [5]:
tokenized_corpus = [doc.split() for doc in documents]

print("Tokenized corpus thành công.")
print("Số document:", len(tokenized_corpus))
print("Ví dụ 30 token đầu:")
print(tokenized_corpus[0][:30])

Tokenized corpus thành công.
Số document: 160254
Ví dụ 30 token đầu:
['tên', 'cướp', 'tiệm', 'vàng', 'tại', 'huế', 'là', 'đại_uý', 'công_an', 'công_tác', 'tại', 'trại_giam', 'chiều', 'date0731', 'công_an', 'tỉnh', 'thừa_thiên', '-', 'huế', 'đã', 'có', 'thông_tin', 'ban_đầu', 'về', 'vụ', 'nổ_súng', 'cướp', 'tiệm', 'vàng', 'tại']


Xây dựng BM25 model

In [6]:
bm25_model = BM25Okapi(tokenized_corpus)

print("Đã xây dựng BM25 model thành công.")

Đã xây dựng BM25 model thành công.


Hàm tìm kiếm BM25

In [7]:
def search_bm25(query_text, k, bm25_model, df_documents):
    """
    Tìm kiếm document bằng thuật toán BM25.

    Input:
        query_text: câu truy vấn người dùng
        k: số lượng kết quả muốn lấy
        bm25_model: mô hình BM25 đã xây dựng
        df_documents: DataFrame chứa dữ liệu bài báo

    Output:
        DataFrame gồm top-k bài báo có điểm BM25 cao nhất
    """

    tokenized_query = query_text.split()

    scores = bm25_model.get_scores(tokenized_query)

    top_k_idx = np.argsort(scores)[::-1][:k]

    results = df_documents.iloc[top_k_idx].copy()
    results["bm25_score"] = scores[top_k_idx]

    return results

Test BM25 Search

In [8]:
query = "bóng đá việt nam"
k_results = 5

results = search_bm25(
    query_text=query,
    k=k_results,
    bm25_model=bm25_model,
    df_documents=df_load
)

results[["title", "source", "topic", "bm25_score"]].head(k_results)

,title,source,topic,bm25_score
132485,Ngày hội thiếu nhi và gia đình làng Việt tại W...,baoquocte,Ngoại giao,14.307397
117685,Giao hữu ĐT nữ Việt Nam – ĐT nữ Pháp: Khi thắn...,vov.vn,THỂ THAO,13.785753
116887,Giao hữu ĐT nữ Việt Nam – ĐT nữ Pháp: Khi thắn...,soha,,13.785753
151660,Quang Hải hài lòng với CLB nhỏ ở Pháp,vnexpress,Thể thao,13.646628
142057,Nguyễn Quang Hải đầu quân cho Pau FC - VnExpre...,vnexpress,Thể thao,13.441839


Hiển thị kết quả

In [9]:
def display_bm25_results(results):
    for _, row in results.iterrows():
        print("=" * 100)
        print("Tiêu đề:", row.get("title", "Không có title"))
        print("Điểm BM25:", row["bm25_score"])
        print("Tác giả:", row.get("author", "Không có author"))
        print("Nguồn:", row.get("source", "Không có source"))
        print("Chủ đề:", row.get("topic", "Không có topic"))
        print("URL:", row.get("url", "Không có url"))
        print("Ngày crawl:", row.get("crawled_at", "Không có crawled_at"))

        print("\nNội dung ngắn:")
        print(str(row.get("content", ""))[:700])
        print()

In [10]:
display_bm25_results(results)

Tiêu đề: Ngày hội thiếu nhi và gia đình làng Việt tại Washington D.C, Hoa Kỳ
Điểm BM25: 14.307396754468451
Tác giả: Chu An
Nguồn: baoquocte
Chủ đề: Ngoại giao
URL: https://baoquocte.vn/ngay-hoi-thieu-nhi-va-gia-dinh-lang-viet-tai-washington-dc-hoa-ky-188872.html
Ngày crawl: 2022-06-29 11:51:10.130962

Nội dung ngắn:
Đây là lần đầu tiên sau 2 năm đại dịch Covid-19, các cháu, con của cán bộ ngoại giao thuộc Cơ quan đại diện Việt Nam tại Washington D.C và toàn thể gia đình làng Việt với hơn 130 người, từ các em nhỏ 2-3 tuổi tới ông bà, bố mẹ của cán bộ Việt Nam, được tham dự một buổi gặp gỡ, giao lưu ngoài trời đông vui như thế này. Phát biểu khai mạc buổi giao lưu, Đại sứ Nguyễn Quốc Dũng hoan nghênh ý tưởng tổ chức buổi gặp gỡ giữa toàn thể các gia đình, cho rằng đây là một hoạt động rất ý nghĩa, góp phần tạo nên sự gắn kết gần gũi không chỉ giữa các cán bộ Việt Nam mà còn giữa các thành viên trong mỗi gia đình làng Việt. Mọi người gặp gỡ để cùng chia sẻ, cảm thông, giúp đỡ nhau trong c

Lưu tokenized corpus

In [11]:
corpus_path = os.path.join(save_dir, "tokenized_corpus.pkl")

with open(corpus_path, "wb") as f:
    pickle.dump(tokenized_corpus, f)

print("Đã lưu tokenized corpus tại:", corpus_path)

Đã lưu tokenized corpus tại: indexes/bm25\tokenized_corpus.pkl


Lưu DataFrame đầy đủ

In [12]:
df_path = os.path.join(save_dir, "news_df_full.pkl")

df_load.to_pickle(df_path)

print("Đã lưu DataFrame đầy đủ tại:", df_path)

Đã lưu DataFrame đầy đủ tại: indexes/bm25\news_df_full.pkl


Load lại corpus và build lại BM25

In [13]:
# Cell này dùng cho lần mở notebook sau.
corpus_path = os.path.join(save_dir, "tokenized_corpus.pkl")
df_path = os.path.join(save_dir, "news_df_full.pkl")

df_load = pd.read_pickle(df_path)

with open(corpus_path, "rb") as f:
    tokenized_corpus = pickle.load(f)

bm25_model = BM25Okapi(tokenized_corpus)

print("Load corpus và build lại BM25 thành công.")
print("df_load shape:", df_load.shape)
print("Số document:", len(tokenized_corpus))

Load corpus và build lại BM25 thành công.
df_load shape: (160254, 14)
Số document: 160254


Test lại sau khi load

In [14]:
query = "kinh tế việt nam"
k_results = 10

results = search_bm25(
    query_text=query,
    k=k_results,
    bm25_model=bm25_model,
    df_documents=df_load
)

results[["title", "source", "topic", "bm25_score"]].head(k_results)

,title,source,topic,bm25_score
84560,Lý giải về nghi thức 'Thông dạ tế' trong lễ ta...,thanhnien.vn,Văn hóa,18.065767
109239,Các vua triều Nguyễn tưởng niệm ngày kinh thàn...,danviet,Đông Tây - Kim Cổ,16.848286
2469,Phát hiện dấu tích nghi là đàn tế thời vua Qua...,vnexpress,Thời sự,15.956354
6751,"Huế: Phát lộ cấu trúc, quy mô của đàn tế Giao ...",laodong,Văn hóa - Giải trí,15.894430
6394,Tìm thấy dấu tích đàn Nam Giao thời Tây Sơn ở Huế,thanhnien.vn,Văn hóa,15.502733
6762,Phát hiện bất ngờ về dấu tích Đàn tế cáo trời ...,tienphong,Văn hóa,15.180722
156936,Lễ tế đàn Âm hồn tưởng nhớ ngày kinh đô Huế th...,vnexpress,Thời sự,14.801940
157036,Lễ tế tưởng nhớ ngày kinh đô Huế thất thủ - Vn...,vnexpress,Thời sự,14.754598
89482,Kiếm tìm nguyên gốc Bân Sơn,laodong,Lao Động cuối tuần,14.701741
11405,"Đảm bảo quyền tự do tín ngưỡng, tôn giáo cho 1...",vov.vn,XÃ HỘI,14.500260


In [15]:
import pickle
import os

# Đường dẫn lưu file
bm25_model_path = os.path.join(save_dir, "bm25_model_pretrained.pkl")

# Dump toàn bộ object mô hình BM25 xuống ổ cứng
with open(bm25_model_path, "wb") as f:
    pickle.dump(bm25_model, f)

print(f"Đã lưu trực tiếp toàn bộ mô hình BM25 tại: {bm25_model_path}")

"""
Ghi chú cho lần load sau:
Thay vì phải load corpus và build lại, bạn chỉ cần gọi 1 dòng:
with open("indexes/bm25/bm25_model_pretrained.pkl", "rb") as f:
    bm25_model = pickle.load(f)
"""

Đã lưu trực tiếp toàn bộ mô hình BM25 tại: indexes/bm25\bm25_model_pretrained.pkl


'\nGhi chú cho lần load sau:\nThay vì phải load corpus và build lại, bạn chỉ cần gọi 1 dòng:\nwith open("indexes/bm25/bm25_model_pretrained.pkl", "rb") as f:\n    bm25_model = pickle.load(f)\n'